In [1]:
from paxini_utils.high_speed_board import HighSpeedBoard

In [2]:
board = HighSpeedBoard("/dev/cu.usbmodem750A687485311")

In [3]:
board.open()

In [4]:
board.get_connected_sensors()

BoardStatus(connected_sensors=('ring_near', 'pinky_near'))

In [5]:
board.get_version()

'ADAPTER_FW02.22'

In [6]:
board.get_distribution_point_counts()

{'ring_near': 77, 'pinky_near': 77}

In [7]:
import html
import time
from IPython.display import HTML, clear_output, display

def render_force_readings(readings, frame_index):
    rows = []
    for reading in readings:
        total = reading.total_force
        total_x = total.x if total else 0.0
        total_y = total.y if total else 0.0
        total_z = total.z if total else 0.0
        max_point_z = max((point.force.z for point in reading.distribution), default=0.0)
        active_points = sum(1 for point in reading.distribution if abs(point.force.x) + abs(point.force.y) + abs(point.force.z) > 0)
        bar_width = min(100, max(2, total_z * 8))
        rows.append(f'''
            <tr>
                <td><code>{html.escape(reading.sensor)}</code></td>
                <td>{total_x:6.1f}</td>
                <td>{total_y:6.1f}</td>
                <td>{total_z:6.1f}</td>
                <td>{max_point_z:6.1f}</td>
                <td>{active_points}/{len(reading.distribution)}</td>
                <td><div style="width:160px;background:#eee;border-radius:999px;overflow:hidden"><div style="height:12px;width:{bar_width}%;background:#2563eb"></div></div></td>
            </tr>
        ''')

    return HTML(f'''
        <div style="font-family:system-ui,-apple-system,sans-serif;max-width:900px">
            <h3 style="margin:0 0 8px">High Speed Board Live Forces</h3>
            <div style="color:#666;margin-bottom:10px">Frame {frame_index} - {time.strftime('%H:%M:%S')}</div>
            <table style="border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums">
                <thead>
                    <tr style="text-align:left;border-bottom:1px solid #ddd">
                        <th>Sensor</th><th>Fx N</th><th>Fy N</th><th>Fz N</th><th>Max Point Z</th><th>Active Points</th><th>Total Fz</th>
                    </tr>
                </thead>
                <tbody>{''.join(rows)}</tbody>
            </table>
        </div>
    ''')

refresh_seconds = 0.05

try:
    frame_index = 0
    while True:
        frame_index += 1
        readings = board.read_connected_distribution_forces()
        clear_output(wait=True)
        display(render_force_readings(readings, frame_index))
        time.sleep(refresh_seconds)
except KeyboardInterrupt:
    clear_output(wait=True)
    display(HTML("<b>Stopped.</b>"))


In [96]:
%%time
COUNT=1000
for i in range(COUNT):
    board.read_connected_distribution_forces()

CPU times: user 622 ms, sys: 1.27 s, total: 1.89 s
Wall time: 18.9 s
